In [ ]:
from pathlib import Path

from osgeo import gdal

PATH_CWIV3_ONTARIO = Path("CWIV3-Ontario.tif")
PATH_SAT_IMGS_BASE = Path("2_sat_imgs/")
PATH_GT_BASE = Path("3_groundtruth/")


def generate_groundtruth(gt_path, sat_path, output_path):
    gt_path = str(gt_path)
    sat_path = str(sat_path)
    output_path = str(output_path)

    sat = gdal.Open(sat_path)
    if sat is None:
        raise ValueError(f"Cannot open satellite file: {sat_path}")

    gt = gdal.Open(gt_path)
    if gt is None:
        raise ValueError(f"Cannot open groundtruth file: {gt_path}")

    # Get satellite projection
    target_srs = sat.GetProjection()

    # Get satellite geotransform
    gt_sat = sat.GetGeoTransform()
    x_res = gt_sat[1]
    y_res = abs(gt_sat[5])

    # Get satellite bounds
    minx = gt_sat[0]
    maxy = gt_sat[3]
    maxx = minx + sat.RasterXSize * x_res
    miny = maxy - sat.RasterYSize * y_res

    # Warp options
    warp_options = gdal.WarpOptions(
        format="GTiff",
        outputBounds=(minx, miny, maxx, maxy),
        xRes=x_res,
        yRes=y_res,
        targetAlignedPixels=True,
        dstSRS=target_srs,
        resampleAlg="near",  # VERY IMPORTANT for labels
        creationOptions=["COMPRESS=LZW"]
    )

    # Execute warp
    gdal.Warp(output_path, gt, options=warp_options)

    print("Groundtruth saved to:", output_path)


def build_gt_path(sat_path):
    sat_path = Path(sat_path)

    # Get path relative to satellite base
    relative_path = sat_path.relative_to(PATH_SAT_IMGS_BASE)

    # Change filename to add _gt
    gt_filename = relative_path.stem + "_gt.tif"

    # Rebuild full path under groundtruth base
    gt_path = PATH_GT_BASE / relative_path.parent / gt_filename

    return gt_path


In [ ]:
for usage in ["train", "test"]:
    sat_tiles = (PATH_SAT_IMGS_BASE / usage).glob("*.tif")

    for sat in sat_tiles:
        out = build_gt_path(sat)
        generate_groundtruth(PATH_CWIV3_ONTARIO, sat, out)